# Gold-standard pool triage — the 572 families of the assignee query (8 Nov 2026)

Input: `01_query/patseer_goldstandard_572_2026-11-08.xlsx` (query in `01_query/query_goldstandard.txt`), the thesis export
`01_query/patseer_1639_2026-06-08.xlsx`, `03_aircraft/aircraft_627.csv` and `06_gold_standard/v1/gold_standard_v1.csv`.

Cells, in order:

1. **Companies, query vs dataset (3a)** — the 42 ASN terms classified wingless / winged / not VTOL; families per term in the thesis export (`Current Assignee`, substring, case-insensitive), designs in the 627, patents in gold standard v1.
2. **The pool (3b)** — the same counts on the 572 families; which of the 56 are in the pool (by record number, then by simple family members); overlap pool ∩ thesis dataset.
3. **Keyword pass** — same regexes and verdict rule as `01_query/branchB/branchB_triage.ipynb` (electric / VTOL / passenger / multicopter + helicopter, UAV, wing, component exclusions).
4. **PatentSBERTa pass** — the thesis T1 classifier and the G1 architecture guess (`Patent-Labelling-Tools/src/reviewer.py`), plus the auxiliary domain probe.
5. **Review sheet** — `goldstandard_pool_review.xlsx`, one row per family with the proposals and empty `coder_*` columns (drop-downs); `priority` = 1 marks the families with a contact sheet. Re-running keeps what the coder typed.
6. **Contact sheets** — drawings from Google Patents for the priority-1 families, `sheets/index.html`.
7. **Summary** — verdicts and architectures by company class; families of the pool that are not in the thesis dataset, by verdict.

The method is the branch-B one (`docs/HANDOFF_branchB.md`). The gold standard v1 and its query are **not** changed here; this is the pool from which the design-based v2 will be built.

In [ ]:
import re, sys, json, math, time, html
from pathlib import Path
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")

HERE = Path.cwd().resolve()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / "01_query").is_dir())
REPO = Path("Patent-Labelling-Tools")   # for src.reviewer (PatentSBERTa definitions)
sys.path.insert(0, str(REPO))
OUT = ROOT / "06_gold_standard" / "v2"

POOL_XLSX   = ROOT / "01_query/patseer_goldstandard_572_2026-11-08.xlsx"
THESIS_XLSX = ROOT / "01_query/patseer_1639_2026-06-08.xlsx"
AIRCRAFT    = ROOT / "03_aircraft/aircraft_627.csv"
GOLD_V1     = ROOT / "06_gold_standard/v1/gold_standard_v1.csv"
REVIEW_XLSX = OUT / "goldstandard_pool_review.xlsx"
TABLES_XLSX = OUT / "goldstandard_companies.xlsx"

pool = pd.read_excel(POOL_XLSX); thesis = pd.read_excel(THESIS_XLSX)
air = pd.read_csv(AIRCRAFT, keep_default_na=False)
gold = pd.read_csv(GOLD_V1)
ca = dict(zip(thesis["Record Number"], thesis["Current Assignee"].astype(str)))
gold["assignee"] = [a if isinstance(a, str) else ca.get(p, "") for a, p in zip(gold.assignee, gold["publication number"])]
assert pool["Record Number"].is_unique and pool["Simple Family ID"].is_unique
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 70)
print(f"pool: {len(pool)} families   thesis export: {len(thesis)} families   designs: {len(air)}   gold v1: {len(gold)}")

## 1. Companies of the query, in the thesis dataset (3a)

One row per ASN term of the query, grouped by company. Class per the author: **sem asa** = Volocopter/e-volo, EHang, ZeroG, Lodd, SkyDrive, ERC System, Lift Aircraft, Jetson, Huitian; **não VTOL** = Electra, Heart, Vaeridion, Regent; everything else **com asa**. Matching is a case-insensitive substring (word-bounded where the term is a common word) on `Assignee` **and** `Current Assignee` together, because PatSeer's `ASN:` field searches both and Bell's patents now sit under Textron Innovations as current assignee; a family that matches two terms of the same company is counted once in the class totals.

In [ ]:
ASN = [  # (term as in the query, company group, class, regex on Current Assignee)
 ("BETA AIR", "Beta", "com asa", r"beta air"), ("Beta Technologies", "Beta", "com asa", r"beta technolog"),
 ("Eve Holding", "Embraer / Eve", "com asa", r"eve holding"), ("EVE UAM", "Embraer / Eve", "com asa", r"eve uam"),
 ("Eve Air Mobility", "Embraer / Eve", "com asa", r"eve air mobility"), ("Embraer", "Embraer / Eve", "com asa", r"embraer"),
 ("Airbus", "Airbus", "com asa", r"airbus"), ("Jaunt", "Jaunt", "com asa", r"\bjaunt\b"), ("Jaunt Air", "Jaunt", "com asa", r"jaunt air"),
 ("Bell", "Bell / Textron", "com asa", r"\bbell\b"), ("Joby", "Joby", "com asa", r"\bjoby\b"), ("Archer Aviation", "Archer", "com asa", r"archer aviation"),
 ("Lilium", "Lilium", "com asa", r"lilium"), ("Wisk", "Wisk", "com asa", r"\bwisk\b"),
 ("Volocopter", "Volocopter / e-volo", "sem asa", r"volocopter"), ("e-volo", "Volocopter / e-volo", "sem asa", r"e-volo"),
 ("Ehang", "EHang", "sem asa", r"ehang"), ("Autoflight", "AutoFlight", "com asa", r"autoflight"), ("Aerofugia", "Aerofugia / Geely", "com asa", r"aerofugia"),
 ("Regent", "Regent", "não VTOL", r"\bregent\b"), ("Volant Aerotech", "Volant", "com asa", r"volant aerotech"),
 ("ZeroG", "ZeroG", "sem asa", r"\bzerog\b|zero g\b"), ("Lodd", "Lodd", "sem asa", r"\blodd\b"), ("Moya Aero", "Moya", "com asa", r"moya aero"),
 ("TCab*", "TCab", "com asa", r"\btcab"), ("Vertaxi", "Vertaxi", "com asa", r"vertaxi"), ("SkyDrive", "SkyDrive", "sem asa", r"skydrive"),
 ("Vertical Aerospace", "Vertical Aerospace", "com asa", r"vertical aerospace"), ("Ascendance", "Ascendance", "com asa", r"ascendance"),
 ("Electra", "Electra", "não VTOL", r"\belectra\b"), ("Heart Aerospace", "Heart Aerospace", "não VTOL", r"heart aerospace"),
 ("Robinson", "Robinson", "com asa", r"\brobinson\b"), ("Supernal", "Supernal", "com asa", r"supernal"), ("Honda Motor", "Honda", "com asa", r"honda motor"),
 ("Vaeridion", "Vaeridion", "não VTOL", r"vaeridion"), ("ERC System", "ERC System", "sem asa", r"erc system"),
 ("Kitty Hawk", "Kitty Hawk / Zee.Aero", "com asa", r"kitty hawk"), ("Zee.Aero", "Kitty Hawk / Zee.Aero", "com asa", r"zee\.?aero"),
 ("Huitian", "Huitian", "sem asa", r"huitian"), ("XPeng", "XPeng", "com asa", r"xpeng"), ("Lift Aircraft", "Lift Aircraft", "sem asa", r"lift aircraft"),
 ("Jetson", "Jetson", "sem asa", r"\bjetson\b"),
]
assert len(ASN) == 42
RXA = {t: re.compile(rx, re.I) for t, _, _, rx in ASN}
CLASS = {t: c for t, _, c, _ in ASN}; GROUP = {t: g for t, g, _, _ in ASN}

def match_terms(text):
    text = str(text); return [t for t in RXA if RXA[t].search(text)]

def company_table(fam_df, fam_col, label):
    """one row per ASN term; `label` names the family column in the result"""
    hits = (fam_df["Assignee"].astype(str) + " | " + fam_df[fam_col].astype(str)).map(match_terms)
    air_hits = (air["assignee"] + " | " + air["company"] + " | " + air["firm"]).map(match_terms)
    gold_hits = gold["assignee"].astype(str).map(match_terms)
    rows = []
    for t, g, c, _ in ASN:
        rows.append({"ASN term": t, "empresa": g, "classe": c,
                     label: int(hits.map(lambda L: t in L).sum()),
                     "designs nos 627": int(air_hits.map(lambda L: t in L).sum()),
                     "patentes no gold v1": int(gold_hits.map(lambda L: t in L).sum())})
    tab = pd.DataFrame(rows)
    # class totals on unique families / designs / gold rows (a row matching two terms of a class counts once)
    def cls_total(series, c): return int(series.map(lambda L: any(CLASS[t] == c for t in L)).sum())
    tot = pd.DataFrame([{"ASN term": "TOTAL", "empresa": "", "classe": c, label: cls_total(hits, c),
                         "designs nos 627": cls_total(air_hits, c), "patentes no gold v1": cls_total(gold_hits, c)}
                        for c in ("sem asa", "com asa", "não VTOL")])
    unmatched = int(hits.map(len).eq(0).sum())
    return pd.concat([tab, tot], ignore_index=True), hits, unmatched

tab_thesis, thesis_hits, un_thesis = company_table(thesis, "Current Assignee", "famílias no dataset (1 639)")
print(tab_thesis.to_string(index=False))
print(f"\nthesis-export families matching no ASN term: {un_thesis} of {len(thesis)}")
print("gold v1 rows matching no ASN term:", gold.loc[gold.assignee.astype(str).map(match_terms).map(len).eq(0), "publication number"].tolist())

## 2. The pool of 572 (3b)

Same table on the pool, then: which of the 56 gold-standard patents are in the pool (the pool is the query the gold standard was drawn from, so most should be), and how many pool families are already in the thesis dataset (by `Simple Family ID` and by publication numbers of `Simple Family Members`).

In [ ]:
tab_pool, pool_hits, un_pool = company_table(pool, "Current Assignee", "famílias no pool (572)")
print(tab_pool.to_string(index=False))
print(f"\npool families matching no ASN term on Assignee + Current Assignee: {un_pool} of {len(pool)}")
if un_pool:
    print(pool.loc[pool_hits.map(len).eq(0), ["Record Number", "Assignee", "Current Assignee"]].head(15).to_string(index=False))

PUB_RE = re.compile(r"[A-Z]{2}\d+[A-Z]\d?")
def members(row):
    return set(PUB_RE.findall(str(row["Simple Family Members"]))) | {str(row["Record Number"])}
pool_members = {}
for _, r in pool.iterrows():
    for m in members(r): pool_members[m] = r["Record Number"]
thesis_members = {}
for _, r in thesis.iterrows():
    for m in members(r): thesis_members[m] = r["Record Number"]

norm = lambda s: re.sub(r"[^A-Z0-9]", "", str(s).upper())
base = lambda s: re.sub(r"[A-Z]\d?$", "", norm(s))
pool_base = {base(m): rec for m, rec in pool_members.items()}
gold["in_pool_by_number"] = gold["publication number"].map(lambda g: base(g) in {base(r) for r in pool["Record Number"]})
gold["in_pool_by_family"] = gold["publication number"].map(lambda g: base(g) in pool_base)
gold["pool_record"] = gold["publication number"].map(lambda g: pool_base.get(base(g), ""))
print(f"\ngold v1 in the pool: by record number {int(gold.in_pool_by_number.sum())}/56, by simple family {int(gold.in_pool_by_family.sum())}/56")
print(pd.crosstab(gold["captured as"], gold.in_pool_by_family, margins=True))
print("\ngold patents NOT in the pool:")
print(gold.loc[~gold.in_pool_by_family, ["publication number", "assignee", "captured as"]].to_string(index=False))

thesis_base = {base(m): rec for m, rec in thesis_members.items()}
pool["in_thesis_dataset"] = pool.apply(lambda r: any(base(m) in thesis_base for m in members(r)), axis=1)
pool["thesis_record"] = pool.apply(lambda r: next((thesis_base[base(m)] for m in members(r) if base(m) in thesis_base), ""), axis=1)
pool["asn_class"] = pool_hits.map(lambda L: "/".join(sorted({CLASS[t] for t in L})) or "no ASN match")
pool["asn_company"] = pool_hits.map(lambda L: "/".join(sorted({GROUP[t] for t in L})) or "")
print(f"\npool ∩ thesis dataset: {int(pool.in_thesis_dataset.sum())} of {len(pool)} pool families are in the 1 639")
print(pd.crosstab(pool.asn_class, pool.in_thesis_dataset, margins=True))

with pd.ExcelWriter(TABLES_XLSX) as xw:
    tab_thesis.to_excel(xw, "3a_thesis_dataset", index=False)
    tab_pool.to_excel(xw, "3b_pool_572", index=False)
    gold[["publication number", "assignee", "captured as", "captured through", "in_pool_by_number", "in_pool_by_family", "pool_record"]].to_excel(xw, "gold_v1_in_pool", index=False)
    pool[["Record Number", "Title", "Current Assignee", "asn_company", "asn_class", "Priority Year", "in_thesis_dataset", "thesis_record"]].to_excel(xw, "pool_vs_thesis", index=False)
print("written", TABLES_XLSX.name)

## 3. Keyword pass

Copied from the branch-B notebook: each regex runs on title, abstract, claims, description (in that order of trust); the evidence column keeps the first matching sentence. Verdict rule: **in** = electric, VTOL, passenger and multicopter all found in title/abstract/claims; **similar** = multicopter + VTOL strong and three of four found, or all four found anywhere; **out** otherwise, or helicopter / UAV terms in title-abstract without the counterpart. `wing` and `component` are reported for the coder, not used in the verdict.

In [ ]:
FLAGS = {
 "electric":    r"\belectri\w*\W+(?:\w+\W+){0,2}(?:motor|propuls|driv|power|engine|rotor|fan)\w*|\bevtol\b|battery[- ](?:powered|driven|electric)|\ball[- ]electric\b|distributed(?: electric)? propulsion|\belectric(?:al)?ly[- ]driven",
 "vtol":        r"\bvtol\b|vertical(?:ly)?\W+(?:\w+\W+){0,2}(?:take[- ]?off|landing|lift[- ]?off)|\bhover(?:ing|s)?\b|\brotorcraft\b|\bevtol\b|take[- ]?off and land(?:ing)? vertically",
 "passenger":   r"\bpassengers?\b|\boccupants?\b|\bcabins?\b|\bcockpits?\b|\bmanned\b|\bpilots?\b|\bpiloted\b|\bcrew\b|\bseat(?:s|ing)?\b|\brider\b|\bpersons?\b|\bpeople\b|\bhumans?\b|\bcommuter|\bair taxi|\bflying car|\bpersonal (?:air|flying) vehicle|\bcarry(?:ing)? (?:a |one |two |several )?(?:person|people|human)",
 "multicopter": r"multi-?copter|multi-?rotor|quad-?copter|quad-?rotor|hexa-?copter|hexa-?rotor|octo-?copter|octo-?rotor|tri-?copter|volocopter|(?:three|four|six|eight|twelve|sixteen|eighteen|\b[34689]\b|\b1[0-9]\b|plurality of|multiple|array of|a number of|several)\W+(?:\w+\W+){0,3}(?:rotors?|propellers?|ducted fans?|propulsion units?|lift(?:ing)? units?|thrusters?)",
 "uav":         r"\buavs?\b|\bunmanned\b|\bdrones?\b|pilotless|unpiloted|remotely (?:piloted|controlled|operated)|\buas\b",
 "helicopter":  r"\bmain rotor|\btail rotor|\bcoaxial|\btandem rotor|swashplate|cyclic pitch|collective pitch|\bhelicopter",
 "wing":        r"\bwings?\b|fixed[- ]wing|tilt[- ]?(?:wing|rotor)|\bcanard|\bairfoil",
 "component":   r"\bmethod\b|\bbattery\b|\bmotor assembly|\bblade\b|\blanding gear|\bcontrol(?:ler| system| method)\b|\bcharging|\bcooling|\bsensor",
}
RX = {k: re.compile(v, re.I) for k, v in FLAGS.items()}
NEG_PILOT = re.compile(r"auto-?pilot\w*|pilotless|unpiloted|remotely[- ]piloted|without (?:a )?pilot", re.I)
SENT = re.compile(r"(?<=[.;!?])\s+|\n+")

def fields(r):
    return [("title", r["Title"]), ("abstract", r["Abstract"]),
            ("claims", " ".join(str(r[c]) for c in ("First Claim", "Independent Claims") if pd.notna(r[c]))),
            ("description", " ".join(str(r[c]) for c in ("Summary of Invention", "Description of Drawings", "Description") if pd.notna(r[c])))]

def first_hit(rx, r, strip_neg=False):
    for name, txt in fields(r):
        if not isinstance(txt, str) or not txt.strip(): continue
        t = NEG_PILOT.sub(" ", txt) if strip_neg else txt
        m = rx.search(t)
        if m:
            sents = [s for s in SENT.split(t) if rx.search(s)]
            ev = (sents[0] if sents else t[max(0, m.start()-120): m.end()+120]).strip()
            return name, ev[:300]
    return None, None

def count_hits(rx, r, strip_neg=False):
    n = 0
    for name, txt in fields(r):
        if isinstance(txt, str):
            t = NEG_PILOT.sub(" ", txt) if strip_neg else txt
            n += len(rx.findall(t))
    return n

kw = pool[["Record Number", "Simple Family ID", "asn_company", "asn_class", "in_thesis_dataset"]].copy()
for k, rx in RX.items():
    hits = pool.apply(lambda r: first_hit(rx, r, strip_neg=(k == "passenger")), axis=1)
    kw[f"kw_{k}_field"] = [h[0] for h in hits]
    kw[f"kw_{k}_evidence"] = [h[1] for h in hits]
    kw[f"kw_{k}_n"] = pool.apply(lambda r: count_hits(rx, r, strip_neg=(k == "passenger")), axis=1)

def strength(field): return {"title": "strong", "abstract": "strong", "claims": "medium", "description": "weak", None: "none"}[field]
for k in ("electric", "vtol", "passenger", "multicopter"):
    kw[f"kw_{k}"] = kw[f"kw_{k}_field"].map(strength)

core = ["electric", "vtol", "passenger", "multicopter"]
def kw_verdict(r):
    strong = {k for k in core if r[f"kw_{k}"] in ("strong", "medium")}
    weak   = {k for k in core if r[f"kw_{k}"] == "weak"}
    heli   = r["kw_helicopter_field"] in ("title", "abstract") and r["kw_multicopter_field"] not in ("title", "abstract")
    uav    = r["kw_uav_field"] in ("title", "abstract") and r["kw_passenger_field"] not in ("title", "abstract")
    if heli: return "out (helicopter terms)"
    if uav:  return "out (UAV, no crew in TA)"
    if len(strong) == 4: return "in"
    if "multicopter" in strong and "vtol" in strong | weak and len(strong | weak) >= 3: return "similar"
    if len(strong | weak) == 4: return "similar"
    return "out"
kw["kw_verdict"] = kw.apply(kw_verdict, axis=1)
print(pd.crosstab(kw.asn_class, kw.kw_verdict, margins=True))
print(); print(pd.crosstab(kw.in_thesis_dataset, kw.kw_verdict, margins=True))

## 4. PatentSBERTa pass (thesis method, Subsubsection 3.3.1.2)

Same model, input text (title + abstract + description of drawings) and thresholds as the thesis wizard feed. `sbert_arch` is the G1 top-type guess over the taxonomy definitions; `sbert_arch_flagged` marks a thin margin (top-1 − top-2 < 0.05 or confidence < 0.45). The zero-shot *domain* probe is an ordering aid only.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from src.reviewer import classify_t1_dimensions, _sbert_best, _margin_flag, _G1_TOP_TYPE_DEFS

device = "cuda" if torch.cuda.is_available() else "cpu"
sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa", cache_folder=str(REPO / "models" / "SBERT"), device=device)
print("PatentSBERTa on", device)

def t1_text(r):
    return " ".join(str(r[c]) for c in ("Title", "Abstract", "Description of Drawings") if pd.notna(r[c]))

DOMAIN_DEFS = {
    "crewed multicopter":   "a passenger-carrying electric multirotor aircraft with several fixed lift rotors and no wing, carrying one or more people in a cabin or seat",
    "winged eVTOL":         "a vertical take-off aircraft with wings for cruise flight and rotors or tilting propellers for hover",
    "unmanned drone":       "an unmanned aerial vehicle or drone for cargo, inspection or photography without any person on board",
    "helicopter":           "a conventional helicopter with a main rotor and a tail rotor or coaxial rotors",
    "component or method":  "a component, subsystem, battery, motor, control method or software not describing a complete aircraft",
}
AUTO_T = 0.45
rows = []
for _, r in pool.iterrows():
    txt = t1_text(r)
    t1 = classify_t1_dimensions(txt, sbert)
    confs = [t1[k]["confidence"] for k in ("scope", "t1Field", "t1Target") if t1[k]["value"]]
    avg = float(np.mean(confs)) if confs else None
    auto = True if (len(confs) == 3 and avg >= AUTO_T) else (False if not confs else None)
    g1 = _margin_flag(_sbert_best(txt, _G1_TOP_TYPE_DEFS, sbert))
    dom = _sbert_best(txt, DOMAIN_DEFS, sbert)
    rows.append({"Record Number": r["Record Number"],
                 "sbert_scope": t1["scope"]["value"], "sbert_scope_conf": t1["scope"]["confidence"],
                 "sbert_field": t1["t1Field"]["value"], "sbert_field_conf": t1["t1Field"]["confidence"],
                 "sbert_target": t1["t1Target"]["value"], "sbert_target_conf": t1["t1Target"]["confidence"],
                 "sbert_avg_conf": None if avg is None else round(avg, 4), "sbert_auto_approve": auto,
                 "sbert_arch": g1["value"], "sbert_arch_conf": g1["confidence"], "sbert_arch_margin": g1.get("margin"),
                 "sbert_arch_flagged": bool(g1.get("flagged_ambiguous", False)),
                 "sbert_domain_aux": dom["value"], "sbert_domain_aux_conf": dom["confidence"]})
sb = pd.DataFrame(rows)
print(sb.sbert_auto_approve.value_counts(dropna=False).to_string()); print()
print(pd.crosstab(kw.asn_class, sb.sbert_arch, margins=True)); print()
print(pd.crosstab(kw.asn_class, sb.sbert_domain_aux, margins=True))

## 5. Review sheet

`goldstandard_pool_review.xlsx`, sheet **review**, one row per family, sorted: not in the thesis dataset first, then wingless companies, then by keyword verdict. Coder columns as in the branch-B sheet (`coder_electric / vtol / passenger / multicopter` yes/no/unknown; `coder_in_domain` yes/similar/no; `coder_architecture` taxonomy top type; `coder_design_key` same assignee + same airframe = one key; `coder_notes`). Re-running keeps the coder's entries (merged on `Record Number`).

In [ ]:
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

CODER_COLS = ["coder_electric", "coder_vtol", "coder_passenger", "coder_multicopter",
              "coder_in_domain", "coder_architecture", "coder_design_key",
              "coder_shows_crewed_aircraft", "coder_vfs_design", "coder_in_627", "coder_notes"]
US_RE = re.compile(r"^(US)(\d{4})(\d{6})(A\d)$")
def gp_url(pub):
    m = US_RE.match(str(pub)); pub = f"{m[1]}{m[2]}0{m[3]}{m[4]}" if m else pub
    return f"https://patents.google.com/patent/{pub}/en"

keep = ["Record Number", "Title", "Abstract", "Current Assignee", "Assignee", "Priority Year", "EFAM Earliest Priority Date", "Publication/Issue Date",
        "CPC", "Simple Family Members", "No. of Simple Family Members", "PDF Link", "in_thesis_dataset", "thesis_record", "asn_company", "asn_class"]
review = (pool[keep].merge(kw.drop(columns=["Simple Family ID", "asn_company", "asn_class", "in_thesis_dataset"]), on="Record Number")
                    .merge(sb, on="Record Number"))
review.insert(1, "google_patents", review["Record Number"].map(gp_url))
gold_set = {base(g) for g in gold["publication number"]}
review.insert(2, "in_gold_v1", review["Record Number"].map(lambda r: base(r) in gold_set))
for c in CODER_COLS: review[c] = ""

# priority 1 = the families with a contact sheet (Section 6):
#  (a) wingless companies, not in the thesis dataset, in-or-similar by keywords (all of them)
#  (b) 30 winged-company families, not in the thesis dataset, in-or-similar by keywords, random sample, seed 42
cand = review[(~review.in_thesis_dataset) & review.kw_verdict.isin(["in", "similar"])]
grp_a = cand[cand.asn_class == "sem asa"]["Record Number"]
pool_b = cand[cand.asn_class == "com asa"].sort_values("Record Number")
grp_b = pool_b.sample(n=30, random_state=42)["Record Number"]
review.insert(3, "priority", review["Record Number"].map(lambda r: 1 if r in set(grp_a) | set(grp_b) else ""))
review.insert(4, "priority_group", review["Record Number"].map(lambda r: "a: sem asa, fora das 1 639" if r in set(grp_a) else ("b: com asa, amostra 30 (seed 42)" if r in set(grp_b) else "")))
print(f"priority 1: (a) {len(grp_a)} wingless + (b) {len(grp_b)} of {len(pool_b)} winged = {int((review.priority == 1).sum())} families")

if REVIEW_XLSX.exists():   # keep what the coder typed
    old = pd.read_excel(REVIEW_XLSX, sheet_name="review", dtype=str)
    old = old[["Record Number"] + [c for c in CODER_COLS if c in old.columns]].fillna("")
    review = review.drop(columns=CODER_COLS).merge(old, on="Record Number", how="left")
    for c in CODER_COLS: review[c] = review[c].fillna("") if c in review.columns else ""
    print("kept coder columns from the existing sheet:", int((review[CODER_COLS] != "").any(axis=1).sum()), "rows coded")

# coding done in sheets/index.html: the page exports goldstandard_pool_coded_<stamp>.xlsx (browser Downloads).
# The newest such file found here or in ~/Downloads is merged in; its non-empty cells win.
# a file named ..._decisions.xlsx holds decisions the coder gave outside the page (dictated, 2026-11-09); it is a full copy of the 51 rows.
coded_files = sorted(list(OUT.glob("goldstandard_pool_coded_*.xlsx")) + list((Path.home() / "Downloads").glob("goldstandard_pool_coded_*.xlsx")), key=lambda f: f.stat().st_mtime)
if coded_files:
    import shutil
    src = coded_files[-1]
    if src.parent != OUT: shutil.copy(src, OUT / src.name)
    cd = pd.read_excel(src, dtype=str).fillna("").set_index("Record Number")
    n = 0
    for c in [c for c in CODER_COLS if c in cd.columns]:
        m = review["Record Number"].map(cd[c]).fillna("")
        n += int((m != "").sum()); review[c] = np.where(m != "", m, review[c])
    print(f"merged {n} coder values from {src.name}")

# goldstandard_pool_resolution.csv: the coder's free-text notes read by hand (the notebook does not parse notes).
#  - "X é igual" -> same_design_as = X, one coder_design_key per group; blank coder cells of those rows inherit the
#    answers given on X, and inherited_from records it.
#  - rows the coder could not settle from the drawings (electric? carries people?) -> text_* proposals from the
#    patent text, kept in their own columns; the coder columns stay as typed.
RES_COLS = ["same_design_as", "inherited_from", "text_electric", "text_passenger", "text_in_domain", "text_confidence", "text_evidence"]
for c in RES_COLS: review[c] = ""
RES = OUT / "goldstandard_pool_resolution.csv"
if RES.exists():
    res = pd.read_csv(RES, dtype=str).fillna("").set_index("Record Number")
    ix = review.set_index("Record Number").index
    PROP = [c for c in res.columns if c.startswith("prop_")]   # proposals for the coder to confirm (in 627?, VFS name, missing architecture)
    for c in ["same_design_as"] + [c for c in RES_COLS if c.startswith("text_")] + PROP:
        review[c] = ix.map(res[c]).fillna("")
    k = ix.map(res["design_key"]).fillna("")
    review["coder_design_key"] = np.where((review.coder_design_key == "") & (k != ""), k, review.coder_design_key)
    byrec = review.set_index("Record Number")
    INHERIT = ["coder_shows_crewed_aircraft", "coder_architecture", "coder_vfs_design", "coder_in_627", "coder_in_domain"]
    n_inh = 0
    for i, r in review[review.same_design_as != ""].iterrows():
        got = [c for c in INHERIT if r[c] == "" and byrec.at[r.same_design_as, c] != ""]
        for c in got: review.at[i, c] = byrec.at[r.same_design_as, c]
        if got: review.at[i, "inherited_from"] = r.same_design_as; n_inh += 1
    print(f"resolution: {int((review.same_design_as != '').sum())} rows point to another family, {n_inh} inherited its answers, "
          f"{int((review.text_in_domain != '').sum())} rows with text-based proposals")


VORDER = {"in": 0, "similar": 1, "out": 2, "out (UAV, no crew in TA)": 3, "out (helicopter terms)": 4}
review["_s"] = review.kw_verdict.map(VORDER)
review["_p"] = (review.priority != 1).astype(int)
review = (review.sort_values(["_p", "priority_group", "asn_company", "EFAM Earliest Priority Date", "in_thesis_dataset", "asn_class", "_s"],
                             ascending=[True, True, True, True, True, False, True])
                .drop(columns=["_s", "_p"]).reset_index(drop=True))

with pd.ExcelWriter(REVIEW_XLSX, engine="openpyxl") as xw:
    review.to_excel(xw, "review", index=False)
    pd.DataFrame([{"flag": k, "regex": v} for k, v in FLAGS.items()]).to_excel(xw, "regexes", index=False)
    pd.DataFrame([
        {"column": "coder_electric / coder_vtol / coder_passenger / coder_multicopter", "values": "yes / no / unknown", "meaning": "the four domain flags"},
        {"column": "coder_in_domain", "values": "yes / similar / no", "meaning": "yes = all four flags yes; similar = morphology of the domain, uncrewed or not all-electric; no"},
        {"column": "coder_architecture", "values": "MR, HB, PFV, RC, SLC, TR, CVT, TW, TB, PTC, SRW, DS, Other, none", "meaning": "taxonomy top type (MR = wingless multicopter)"},
        {"column": "coder_design_key", "values": "free text", "meaning": "same assignee + same airframe in the drawings = one key; blank = a design of its own"},
        {"column": "coder_shows_crewed_aircraft", "values": "s / n", "meaning": "the drawings show a complete aircraft that carries people"},
        {"column": "coder_vfs_design", "values": "name", "meaning": "the VFS directory design the drawings show, if recognisable"},
        {"column": "coder_in_627", "values": "s / n / ?", "meaning": "that design is already one of the 627 of the thesis"},
        {"column": "priority", "values": "1 / blank", "meaning": "1 = has a contact sheet in sheets/ (group in priority_group)"},
        {"column": "coder_notes", "values": "free text", "meaning": ""},
        {"column": "same_design_as / inherited_from", "values": "Record Number", "meaning": "from goldstandard_pool_resolution.csv: the coder noted the family shows the same design as that record; blank coder cells were copied from it"},
        {"column": "text_electric / text_passenger / text_in_domain / text_confidence / text_evidence", "values": "proposal", "meaning": "read from the patent text where the drawings did not settle electric / carries people; a proposal, not a coder decision"},
        {"column": "prop_shows_crewed / prop_architecture / prop_vfs_design / prop_in_627 / prop_627_record / prop_confidence / prop_note", "values": "proposal", "meaning": "from goldstandard_pool_resolution.csv: proposals for cells the coder left open or marked '?', from the drawings, the patent text and aircraft_627.csv; to be confirmed by the coder, never copied into coder_* automatically"}]).to_excel(xw, "codebook", index=False)

wb = load_workbook(REVIEW_XLSX); ws = wb["review"]
hdr = {c.value: i + 1 for i, c in enumerate(ws[1])}
yellow = PatternFill("solid", fgColor="FFF2CC")
for c in CODER_COLS:
    col = get_column_letter(hdr[c]); ws[f"{col}1"].fill = yellow; ws[f"{col}1"].font = Font(bold=True)
    for row in range(2, ws.max_row + 1): ws[f"{col}{row}"].fill = yellow
def dv(col, formula):
    d = DataValidation(type="list", formula1=formula, allow_blank=True); ws.add_data_validation(d)
    d.add(f"{get_column_letter(hdr[col])}2:{get_column_letter(hdr[col])}{ws.max_row}")
for c in CODER_COLS[:4]: dv(c, '"yes,no,unknown"')
dv("coder_in_domain", '"yes,similar,no"'); dv("coder_shows_crewed_aircraft", '"s,n"'); dv("coder_in_627", '"s,n,?"'); dv("coder_architecture", '"MR,HB,PFV,RC,SLC,TR,CVT,TW,TB,PTC,SRW,DS,Other,none"')
for i, c in enumerate(ws[1], 1):
    ws.column_dimensions[get_column_letter(i)].width = 14 if c.value not in ("Title", "Abstract") and not str(c.value).endswith("evidence") else 45
ws.freeze_panes = "F2"
for row in range(2, ws.max_row + 1):
    cell = ws.cell(row=row, column=hdr["google_patents"]); cell.hyperlink = cell.value; cell.font = Font(color="0563C1", underline="single")
wb.save(REVIEW_XLSX)
print("written", REVIEW_XLSX.name, review.shape)

## 6. Contact sheets (priority 1 only)

For each priority-1 family: the Google Patents page of the record (falling back to other family members) → up to six drawing pages → one PNG grid in `sheets/<Record Number>.png`; CN records that expose only a PDF are rendered from it. `sheets/index.html` lists them by group, company and earliest priority date. Sheets already built are skipped, so the cell can be re-run after a network hiccup.

`sheets/index.html` is an interactive coding page: answers are kept in the browser and **Exportar Excel** downloads `goldstandard_pool_coded_<stamp>.xlsx`; cell 5 merges the newest such file (from this folder or `~/Downloads`) into the review sheet on the next run.

In [ ]:
import requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont

GP_HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
CDN_RE = re.compile(r'https://patentimages\.storage\.googleapis\.com/[^\s"\'<>]+\.png')
SHEETS = OUT / "sheets"; SHEETS.mkdir(exist_ok=True)
RAW = SHEETS / "_raw"; RAW.mkdir(exist_ok=True)
DELAY, MAX_FIGS, THUMB = 0.7, 6, (420, 420)
PRIORITY = ("US", "EP", "WO", "CN", "KR", "JP", "DE", "GB", "FR")

def norm_us(pub):
    m = US_RE.match(pub); return f"{m[1]}{m[2]}0{m[3]}{m[4]}" if m else pub

PDF_RE = re.compile(r'https://patentimages\.storage\.googleapis\.com/[^\s"\'<>]+\.pdf')
import subprocess, tempfile, shutil

def get_with_retry(url, tries=3):
    # Google Patents throttles bursts: back off on 429/5xx or a network error.
    for i in range(tries):
        try:
            r = requests.get(url, headers=GP_HEADERS, timeout=40)
            if r.status_code == 200: return r
            if r.status_code == 404: return None
        except Exception: pass
        time.sleep(5 * (3 ** i))
    return None

def drawing_urls(pub):
    # (png drawing urls, pdf url) for one publication, or ([], None)
    r = get_with_retry(f"https://patents.google.com/patent/{norm_us(pub)}/en")
    if r is None: return [], None
    seen = {}
    for u in CDN_RE.findall(r.text): seen[u.rsplit("/", 1)[-1]] = u
    pdfs = PDF_RE.findall(r.text)
    return [u for _, u in sorted(seen.items())], (pdfs[0] if pdfs else None)

def figs_from_pdf(pdf_url, out_dir, max_figs=MAX_FIGS):
    # CN (and some other) records expose only the PDF: render its drawing pages.
    # A drawing page is one with little extractable text (< 250 chars); CN drawings sit at the end.
    r = get_with_retry(pdf_url)
    if r is None: return []
    tmp = Path(tempfile.mkdtemp()); pdf = tmp / "p.pdf"; pdf.write_bytes(r.content)
    try:
        npages = int(re.search(r"Pages:\s+(\d+)", subprocess.run(["pdfinfo", str(pdf)], capture_output=True, text=True).stdout).group(1))
    except Exception: npages = 0
    pages, total_chars = [], 0
    for p in range(1, npages + 1):
        txt = subprocess.run(["pdftotext", "-f", str(p), "-l", str(p), str(pdf), "-"], capture_output=True, text=True).stdout
        total_chars += len(txt.strip())
        if p > 1 and len(txt.strip()) < 250: pages.append(p)
    if total_chars < 250 * max(1, npages // 2):
        # scanned PDF (no text layer, typical of CN): tell drawing pages from text pages by layout.
        # Text rows span the full column width; drawing rows do not. A page where fewer than 25 %
        # of the inked rows are wide (> 70 % of the body width) is a drawing page. Page 1 is the front page.
        subprocess.run(["pdftoppm", "-r", "40", "-gray", "-png", str(pdf), str(tmp / "lo")], capture_output=True)
        pages = []
        for p, f in enumerate(sorted(tmp.glob("lo-*.png")), 1):
            if p == 1: continue
            a = np.array(Image.open(f).convert("L")); h, w = a.shape
            body = a[int(h * 0.10):int(h * 0.93), int(w * 0.08):int(w * 0.92)] < 140
            ext = []
            for row in body:
                idx = np.flatnonzero(row)
                if len(idx) > 3: ext.append((idx[-1] - idx[0]) / body.shape[1])
            if ext and (np.array(ext) > 0.7).mean() < 0.25: pages.append(p)
    saved = []
    for i, p in enumerate(pages[:max_figs], 1):
        dest = out_dir / f"fig_{i:02d}"
        subprocess.run(["pdftoppm", "-f", str(p), "-l", str(p), "-r", "60", "-png", "-singlefile", str(pdf), str(dest)], capture_output=True)
        if dest.with_suffix(".png").exists(): saved.append(dest.with_suffix(".png"))
    shutil.rmtree(tmp, ignore_errors=True)
    return saved

def family_candidates(row):
    pubs = [row["Record Number"]] + [p for p in PUB_RE.findall(str(row["Simple Family Members"])) if p != row["Record Number"]]
    return sorted(dict.fromkeys(pubs), key=lambda p: (p != row["Record Number"], PRIORITY.index(p[:2]) if p[:2] in PRIORITY else 99))

def fetch_figs(row):
    out_dir = RAW / row["Record Number"]; out_dir.mkdir(exist_ok=True)
    have = sorted(out_dir.glob("fig_*.png"))
    if have: return have, (out_dir / "source.txt").read_text() if (out_dir / "source.txt").exists() else "cache"
    pdf_fallback = None
    for pub in family_candidates(row)[:4]:
        urls, pdf_url = drawing_urls(pub); time.sleep(DELAY)
        if pdf_url and pdf_fallback is None: pdf_fallback = (pub, pdf_url)
        if not urls: continue
        saved = []
        for i, u in enumerate(urls[:MAX_FIGS], 1):
            img = get_with_retry(u)
            if img is not None:
                p = out_dir / f"fig_{i:02d}.png"; p.write_bytes(img.content); saved.append(p)
            time.sleep(DELAY / 2)
        if saved: (out_dir / "source.txt").write_text(pub); return saved, pub
    if pdf_fallback:                       # no PNG drawings anywhere in the family: render the PDF
        pub, pdf_url = pdf_fallback
        saved = figs_from_pdf(pdf_url, out_dir)
        if saved: (out_dir / "source.txt").write_text(pub + " (pdf)"); return saved, pub + " (pdf)"
    return [], None

def font(size):
    for f in ("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"):
        if Path(f).exists(): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def build_sheet(row, figs, src):
    cols, rows_ = 3, 2; W, H = THUMB; pad, head = 10, 70
    sheet = Image.new("RGB", (cols * (W + pad) + pad, head + rows_ * (H + pad) + pad), "white")
    d = ImageDraw.Draw(sheet)
    d.text((pad, 8), f"{row['Record Number']}  |  {row['asn_company']}  |  prio. {str(row['EFAM Earliest Priority Date'])[:10]}  |  kw {row['kw_verdict']}  |  sbert {row['sbert_arch']}  |  drawings from {src or '—'}", fill="black", font=font(18))
    d.text((pad, 34), f"{str(row['Assignee'])[:60]}  |  {str(row['Title'])[:110]}", fill="black", font=font(15))
    for i, p in enumerate(figs[:cols * rows_]):
        try:
            im = Image.open(p).convert("RGB"); im.thumbnail(THUMB)
            x = pad + (i % cols) * (W + pad); y = head + (i // cols) * (H + pad)
            sheet.paste(im, (x + (W - im.width) // 2, y + (H - im.height) // 2))
        except Exception: pass
    if not figs: d.text((pad, head + 20), "no drawings found on Google Patents for this family", fill="red", font=font(18))
    sheet.save(SHEETS / f"{row['Record Number']}.png", optimize=True)


rv = pd.read_excel(REVIEW_XLSX, sheet_name="review")
todo = rv[rv.priority == 1].merge(pool[["Record Number", "Simple Family ID"]], on="Record Number")
log = []
for i, (_, row) in enumerate(todo.iterrows(), 1):
    dest = SHEETS / f"{row['Record Number']}.png"
    if dest.exists(): log.append((row["Record Number"], "cached", None)); continue
    figs, src = fetch_figs(row); build_sheet(row, figs, src)
    log.append((row["Record Number"], len(figs), src))
    if i % 10 == 0: print(f"  {i}/{len(todo)} sheets")
log = pd.DataFrame(log, columns=["Record Number", "figs", "source_pub"])
nofig = [r for r in todo["Record Number"] if not sorted((RAW / r).glob("fig_*.png"))]
print(f"sheets: {len(todo)}   without drawings: {len(nofig)} {nofig}")

PAGE = '<!doctype html><html lang="pt"><head><meta charset="utf-8"><title>Gold-standard pool — codificação</title>\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<script src="https://cdnjs.cloudflare.com/ajax/libs/xlsx/0.18.5/xlsx.full.min.js"></script>\n<style>\n:root{--bg:#f6f7f9;--card:#fff;--line:#d5d9e0;--txt:#1c2330;--mut:#5b6575;--acc:#1f6feb;--ok:#1a7f37;--warn:#b35900}\n*{box-sizing:border-box}body{font-family:system-ui,sans-serif;margin:0;background:var(--bg);color:var(--txt)}\nheader{position:sticky;top:0;z-index:5;background:#fff;border-bottom:1px solid var(--line);padding:10px 16px;display:flex;gap:12px;align-items:center;flex-wrap:wrap}\nheader h1{font-size:16px;margin:0 8px 0 0}button,.btn{font:inherit;padding:6px 12px;border:1px solid var(--line);border-radius:6px;background:#fff;cursor:pointer}\nbutton.primary{background:var(--acc);color:#fff;border-color:var(--acc)}#prog{font-variant-numeric:tabular-nums;color:var(--mut)}\nmain{max-width:1500px;margin:0 auto;padding:12px 16px 80px}h2{font-size:15px;margin:22px 0 6px;color:var(--mut)}\n.card{background:var(--card);border:1px solid var(--line);border-left:5px solid var(--line);border-radius:8px;margin:12px 0;padding:12px;display:grid;grid-template-columns:minmax(0,1fr) 330px;gap:14px}\n.card.done{border-left-color:var(--ok)}.card.partial{border-left-color:var(--warn)}\n.card img{max-width:100%;border:1px solid #eee;cursor:zoom-in}.meta{font-size:13px;color:var(--mut);margin:2px 0 8px}.ttl{font-weight:600}\n.form{display:flex;flex-direction:column;gap:9px;font-size:13px;position:sticky;top:64px;align-self:start}\n.q{display:flex;flex-direction:column;gap:3px}.q>span{font-weight:600}.seg{display:flex;gap:4px;flex-wrap:wrap}\n.seg label{border:1px solid var(--line);border-radius:5px;padding:4px 9px;cursor:pointer;background:#fff;user-select:none}\n.seg input{display:none}.seg input:checked+b{color:#fff}.seg label:has(input:checked){background:var(--acc);border-color:var(--acc);color:#fff}.seg b{font-weight:500}\ninput[type=text],textarea,select{font:inherit;padding:5px 7px;border:1px solid var(--line);border-radius:5px;width:100%}textarea{min-height:44px;resize:vertical}\n.prop{font-size:13px;background:#fff8e1;border:1px solid #f0d98a;border-radius:6px;padding:6px 9px;margin:0 0 8px}.dim{opacity:.45}.hide{display:none!important}#zoom{position:fixed;inset:0;background:rgba(0,0,0,.85);display:none;z-index:9;overflow:auto;cursor:zoom-out}#zoom img{display:block;margin:10px auto;max-width:none;background:#fff}\n@media(max-width:900px){.card{grid-template-columns:1fr}.form{position:static}}\n</style></head><body>\n<header><h1>Gold-standard pool — 51 famílias</h1><span id="prog"></span>\n<label><input type="checkbox" id="todo"> só por fazer</label>\n<button class="primary" id="exp">Exportar Excel</button>\n<label class="btn">Carregar Excel anterior<input type="file" id="imp" accept=".xlsx" style="display:none"></label>\n<button id="clr">Limpar tudo</button><span id="msg" style="color:var(--mut);font-size:13px"></span></header>\n<main id="main"></main><div id="zoom"><img></div>\n<script>\nconst DATA = __DATA__;\nconst KEY = "gs_pool_coder_v1";\nconst COLS = ["coder_shows_crewed_aircraft","coder_architecture","coder_vfs_design","coder_in_627","coder_design_key","coder_in_domain","coder_notes"];\nconst ARCH = ["MR","HB","PFV","RC","SLC","TR","CVT","TW","TB","PTC","SRW","DS","Other","none"];\nlet st = {}; try { st = JSON.parse(localStorage.getItem(KEY) || "{}"); } catch(e) {}\nconst save = () => { try { localStorage.setItem(KEY, JSON.stringify(st)); } catch(e) {} };\nconst esc = s => String(s ?? "").replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",\'"\':"&quot;"}[c]));\nconst get = (id, c) => (st[id] || {})[c] || "";\nfunction seg(id, col, opts){ return `<div class="seg">` + opts.map(o => `<label><input type="radio" name="${col}|${id}" value="${o}" ${get(id,col)===o?"checked":""}><b>${o}</b></label>`).join("") + `</div>`; }\nfunction card(r){ const id = r.rec; return `<div class="card" id="c_${id}" data-id="${id}">\n <div><div class="ttl">${esc(id)} — ${esc(r.title)}</div>\n <div class="meta">${esc(r.company)} · ${esc(r.assignee)} · prioridade ${esc(r.date)} · kw: ${esc(r.kw)} · sbert: ${esc(r.arch)} · <a href="${esc(r.url)}" target="_blank">Google Patents</a>${r.pdf?` · <a href="${esc(r.pdf)}" target="_blank">PDF PatSeer</a>`:""}</div>\n ${r.prop?`<div class="prop"><b>Proposta (por confirmar):</b> ${esc(r.prop)}</div>`:""}\n <img loading="lazy" src="${esc(id)}.png" alt=""></div>\n <div class="form">\n  <div class="q"><span>1. Mostra uma aeronave inteira que leva pessoas?</span>${seg(id,"coder_shows_crewed_aircraft",["s","n"])}</div>\n  <div class="dep">\n  <div class="q"><span>2. Arquitectura</span>${seg(id,"coder_architecture",ARCH)}</div>\n  <div class="q"><span>Design VFS (nome, se reconheceres)</span><input type="text" data-col="coder_vfs_design" value="${esc(get(id,"coder_vfs_design"))}"></div>\n  <div class="q"><span>3. Esse design já está nos 627?</span>${seg(id,"coder_in_627",["s","n","?"])}</div>\n  <div class="q"><span>Chave de design (igual = mesma aeronave)</span><input type="text" data-col="coder_design_key" list="keys" value="${esc(get(id,"coder_design_key"))}"></div>\n  <div class="q"><span>No domínio?</span>${seg(id,"coder_in_domain",["yes","similar","no"])}</div>\n  </div>\n  <div class="q"><span>Notas</span><textarea data-col="coder_notes">${esc(get(id,"coder_notes"))}</textarea></div>\n </div></div>`; }\nfunction status(id){ const s = st[id] || {}; const a = s.coder_shows_crewed_aircraft;\n  if (a === "n") return "done"; if (a === "s") return (s.coder_architecture && s.coder_in_627) ? "done" : "partial";\n  return Object.values(s).some(v => v) ? "partial" : ""; }\nfunction refresh(id){ const el = document.getElementById("c_" + id); if (!el) return; const s = status(id);\n  el.classList.toggle("done", s === "done"); el.classList.toggle("partial", s === "partial");\n  el.querySelector(".dep").classList.toggle("dim", get(id,"coder_shows_crewed_aircraft") === "n");\n  el.classList.toggle("hide", document.getElementById("todo").checked && s === "done"); }\nfunction progress(){ const n = DATA.filter(r => status(r.rec) === "done").length; document.getElementById("prog").textContent = `${n} / ${DATA.length} feitas`;\n  const keys = [...new Set(Object.values(st).map(s => s.coder_design_key).filter(Boolean))].sort();\n  document.getElementById("keys").innerHTML = keys.map(k => `<option value="${esc(k)}">`).join(""); }\nfunction render(){ const groups = [...new Set(DATA.map(r => r.group))]; let h = `<datalist id="keys"></datalist>`;\n  for (const g of groups){ const sub = DATA.filter(r => r.group === g); h += `<h2>${esc(g)} (${sub.length})</h2>` + sub.map(card).join(""); }\n  document.getElementById("main").innerHTML = h; DATA.forEach(r => refresh(r.rec)); progress(); }\nfunction setv(id, col, v){ (st[id] = st[id] || {})[col] = v; save(); refresh(id); progress(); }\ndocument.addEventListener("change", e => { const c = e.target.closest(".card"); if (!c) return; const id = c.dataset.id;\n  if (e.target.type === "radio") setv(id, e.target.name.split("|")[0], e.target.value); else if (e.target.dataset.col) setv(id, e.target.dataset.col, e.target.value.trim()); });\ndocument.addEventListener("click", e => { // click a selected radio again to clear it\n  const l = e.target.closest(".seg label"); if (l){ const i = l.querySelector("input"); const c = l.closest(".card"); const col = i.name.split("|")[0];\n    if (get(c.dataset.id, col) === i.value){ e.preventDefault(); i.checked = false; setv(c.dataset.id, col, ""); } return; }\n  if (e.target.matches(".card img")){ const z = document.getElementById("zoom"); z.firstChild.src = e.target.src; z.style.display = "block"; }\n  else if (e.target.closest("#zoom")) document.getElementById("zoom").style.display = "none"; });\ndocument.getElementById("todo").onchange = () => DATA.forEach(r => refresh(r.rec));\ndocument.getElementById("clr").onclick = () => { if (confirm("Apagar tudo o que foi preenchido nesta página?")){ st = {}; save(); render(); } };\nfunction rows(){ return DATA.map(r => { const o = {"Record Number": r.rec, priority_group: r.group, company: r.company, title: r.title}; COLS.forEach(c => o[c] = get(r.rec, c)); return o; }); }\ndocument.getElementById("exp").onclick = () => { const stamp = new Date().toISOString().slice(0,16).replace(/[-:T]/g, ""); const name = `goldstandard_pool_coded_${stamp}`;\n  if (window.XLSX){ const wb = XLSX.utils.book_new(); XLSX.utils.book_append_sheet(wb, XLSX.utils.json_to_sheet(rows()), "coded"); XLSX.writeFile(wb, name + ".xlsx"); }\n  else { const R = rows(), H = Object.keys(R[0]); const csv = [H.join(",")].concat(R.map(o => H.map(k => `"${String(o[k]).replace(/"/g,\'""\')}"`).join(","))).join("\\n");\n    const a = document.createElement("a"); a.href = URL.createObjectURL(new Blob(["\ufeff" + csv], {type: "text/csv"})); a.download = name + ".csv"; a.click(); }\n  document.getElementById("msg").textContent = "exportado: " + name + " (pasta Transferências)"; };\ndocument.getElementById("imp").onchange = e => { const f = e.target.files[0]; if (!f || !window.XLSX) return; const fr = new FileReader();\n  fr.onload = ev => { const wb = XLSX.read(new Uint8Array(ev.target.result), {type: "array"}); const R = XLSX.utils.sheet_to_json(wb.Sheets[wb.SheetNames[0]], {defval: ""}); let n = 0;\n    R.forEach(o => { const id = o["Record Number"]; if (!id) return; COLS.forEach(c => { if (o[c] !== undefined && String(o[c]) !== ""){ (st[id] = st[id] || {})[c] = String(o[c]); n++; } }); });\n    save(); render(); document.getElementById("msg").textContent = `carregado: ${n} valores de ${f.name}`; };\n  fr.readAsArrayBuffer(f); };\nrender();\n</script></body></html>\n'
def prop_text(r):
    # proposals from goldstandard_pool_resolution.csv, shown on the card for the coder to confirm
    g = lambda c: (str(r[c]).strip() if c in r.index and pd.notna(r[c]) else "")
    bits = [f"{lab}: {g(c)}" for lab, c in [("aeronave tripulada", "prop_shows_crewed"), ("arquitectura", "prop_architecture"), ("design VFS", "prop_vfs_design"),
                                            ("nos 627", "prop_in_627"), ("registo nos 627", "prop_627_record"),
                                            ("eléctrico (texto)", "text_electric"), ("leva pessoas (texto)", "text_passenger"), ("no domínio (texto)", "text_in_domain"),
                                            ("mesmo design que", "same_design_as")] if g(c)]
    conf = g("prop_confidence") or g("text_confidence")
    notes = " ".join(x for x in [g("prop_note"), g("text_evidence")] if x)
    return " · ".join(bits) + (f" · confiança {conf}" if conf and bits else "") + (f" — {notes}" if notes else "")
data = [{'rec': r['Record Number'], 'title': str(r['Title']), 'company': str(r['asn_company']), 'assignee': str(r['Current Assignee']).strip(),
         'date': str(r['EFAM Earliest Priority Date'])[:10], 'kw': str(r['kw_verdict']), 'arch': str(r['sbert_arch']), 'group': str(r['priority_group']),
         'url': str(r['google_patents']), 'pdf': (str(r['PDF Link']) if str(r['PDF Link']).startswith('http') else ''), 'prop': prop_text(r)} for _, r in todo.iterrows()]
(SHEETS / 'index.html').write_text(PAGE.replace('__DATA__', json.dumps(data, ensure_ascii=False).replace('</', '<\\/')), encoding='utf-8')
print('wrote', SHEETS / 'index.html', '(interactive coder page,', len(data), 'families)')


## 7. Summary for the author

Verdicts and architecture guesses by company class, then the pool families that are **not** in the thesis dataset (the candidates a design-based gold standard v2 could add), by verdict. Everything below is the script proposal; the coder columns decide.

In [ ]:
print("keyword verdict × class"); print(pd.crosstab(review.asn_class, review.kw_verdict, margins=True)); print()
print("SBERT architecture guess × class (in-or-similar by keywords only)")
ins = review[review.kw_verdict.isin(["in", "similar"])]
print(pd.crosstab(ins.asn_class, ins.sbert_arch, margins=True)); print()
notin = review[~review.in_thesis_dataset]
print(f"pool families NOT in the thesis dataset: {len(notin)} of {len(review)}")
print(pd.crosstab(notin.asn_class, notin.kw_verdict, margins=True)); print()
print("… of which in-or-similar by keywords, by company:")
print(notin[notin.kw_verdict.isin(["in", "similar"])].groupby(["asn_class", "asn_company"]).size().to_string()); print()
print("gold v1 rows in the pool, keyword verdict:")
print(pd.crosstab(review[review.in_gold_v1].asn_class, review[review.in_gold_v1].kw_verdict, margins=True))